In [1]:
import pandas as pd
import os
import pathlib

from pathlib import Path
from collections import defaultdict
from typing import List, Dict

In [2]:
def get_results_comb_bench(folder_path: str, num_runs="_100"):
    df_opt = pd.read_csv("../data/JSP/benchmark_results.csv")
    # print(df_opt)

    dmu_df = pd.read_csv(os.path.join(folder_path, f"DMU{num_runs}.csv"))
    # mask = (df_opt == dmu_df["name"]).all(axis=1)
    df_opt_dmu = df_opt[df_opt["dataname"].isin(dmu_df["name"])]
    dmu_df["n_j"] = df_opt_dmu["n_j"].values
    dmu_df["n_m"] = df_opt_dmu["n_m"].values
    dmu_df.to_csv(os.path.join(folder_path, f"DMU{num_runs}_new.csv"), index=False)
    # print(dmu_df.groupby(["n_j", "n_m"]).mean(numeric_only=True))
    la_df = pd.read_csv(os.path.join(folder_path, f"LA{num_runs}.csv"))
    df_opt_la = df_opt[df_opt["dataname"].isin(la_df["name"])]
    la_df["n_j"] = df_opt_la["n_j"].values
    la_df["n_m"] = df_opt_la["n_m"].values
    la_df.to_csv(os.path.join(folder_path, f"LA{num_runs}_new.csv"), index=False)
    ta_df = pd.read_csv(os.path.join(folder_path, f"TA{num_runs}.csv"))
    df_opt_ta = df_opt[df_opt["dataname"].isin(ta_df["name"])]
    ta_df["n_j"] = df_opt_ta["n_j"].values
    ta_df["n_m"] = df_opt_ta["n_m"].values
    ta_df.to_csv(os.path.join(folder_path, f"TA{num_runs}_new.csv"), index=False)
    ta_gap_greedy = ta_df["gap_det_offline"].mean().round(2)
    ta_gap_samp = ta_df["gap_stoch_offline"].mean().round(2)
    la_gap_greedy = la_df["gap_det_offline"].mean().round(2)
    la_gap_samp = la_df["gap_stoch_offline"].mean().round(2)
    dmu_gap_greedy = dmu_df["gap_det_offline"].mean().round(2)
    dmu_gap_samp = dmu_df["gap_stoch_offline"].mean().round(2)
    return ta_gap_greedy, ta_gap_samp, la_gap_greedy, la_gap_samp, dmu_gap_greedy, dmu_gap_samp


def group_folders_by_prefix(parent: str | Path) -> dict[str, list[Path]]:
    """
    Traverse *parent* and group every immediate sub‑directory by the prefix
    that appears before the first hyphen in its name.

    Parameters
    ----------
    parent : str | Path
        Path to the directory whose immediate sub‑folders will be examined.

    Returns
    -------
    dict[str, list[pathlib.Path]]
        Mapping from discovered prefix to the corresponding folders.
    """
    parent = Path(parent).expanduser().resolve()
    if not parent.is_dir():
        raise NotADirectoryError(f"{parent} is not a directory.")

    groups: dict[str, list[Path]] = defaultdict(list)

    for entry in parent.iterdir():
        if entry.is_dir():
            prefix, *_ = entry.name.split("-", 1)
            groups[prefix].append(entry)

    # (Optional) sort the lists for deterministic output
    for folder_list in groups.values():
        folder_list.sort()

    return dict(groups)


def gather_results_method(path_folder: str | Path, large: bool = False) -> dict[str, List]:
    """
    Gather results from all folders in the given path.

    Parameters
    ----------
    path_folder : str | Path
        Path to the directory containing the folders.
    method : str
        Method to gather results for.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary with folder names as keys and DataFrames as values.
    """
    path_folder = os.path.join(path_folder, "sd1")

    path_folder = Path(path_folder).expanduser().resolve()
    if not path_folder.is_dir():
        raise NotADirectoryError(f"{path_folder} is not a directory.")

    folder_dict = group_folders_by_prefix(path_folder)
    results = {}


    for exp_name, folders in folder_dict.items():
        results[exp_name] = {
            "la_gap_greedy": [],
            "la_gap_samp": [],
            "ta_gap_greedy": [],
            "ta_gap_samp": [],
            "dmu_gap_greedy": [],
            "dmu_gap_samp": [],


        }

        assert len(folders) == 4, f"Expected 4 folders for {exp_name}, but found {len(folders)}"
        for folder in folders:
            n_better = 0
            ta_gap_greedy, ta_gap_samp, la_gap_greedy, la_gap_samp, dmu_gap_greedy, dmu_gap_samp = get_results_comb_bench(folder)
            print("folder", folder)
            print("ta_gap_greedy", ta_gap_greedy)
            print("ta_gap_samp", ta_gap_samp)
            print("la_gap_greedy", la_gap_greedy)
            print("la_gap_samp", la_gap_samp)
            print("dmu_gap_greedy", dmu_gap_greedy)
            print("dmu_gap_samp", dmu_gap_samp)
            # if mk_gap_greedy < 12.97:
            #     n_better += 1
            # if mk_gap_samp < 8.95:
            #     n_better += 1
            # if rdata_gap_greedy < 11.15:
            #     n_better += 1
            # if rdata_gap_samp < 4.95:
            #     n_better += 1
            # if edata_gap_greedy < 14.41:
            #     n_better += 1
            # if edata_gap_samp < 8.17:
            #     n_better += 1
            # if vdata_gap_greedy < 3.28:
            #     n_better += 1
            # if vdata_gap_samp < 0.69:
            #     n_better += 1
            # if large:
            #     if gen_gap_greedy < 12.42:
            #         n_better += 1
            #     if gen_gap_samp < 6.79:
            #         n_better += 1
            # else:
            #
            #     if gen_gap_greedy < 10.87:
            #         n_better += 1
            #     if gen_gap_samp < 5.57:
            #         n_better += 1
            # print(f"Folder: {folder}, n_better: {n_better}")



gather_results_method("./checkpoints_exp_jsp/jsp_dataset_exp_15_10/CDQAC", large=False)
# gather_results_method("./checkpoints_exp/dataset_exp_15_10/CDQAC/", large=True)



folder C:\Users\jesse\PycharmProjects\offline_fjsp\offline-fjsp\collect_results\checkpoints_exp_jsp\jsp_dataset_exp_15_10\CDQAC\sd1\dispatching-06ba7d36_seed_2
ta_gap_greedy 16.8
ta_gap_samp 12.53
la_gap_greedy 10.26
la_gap_samp 5.4
dmu_gap_greedy 28.42
dmu_gap_samp 23.26
folder C:\Users\jesse\PycharmProjects\offline_fjsp\offline-fjsp\collect_results\checkpoints_exp_jsp\jsp_dataset_exp_15_10\CDQAC\sd1\dispatching-9e7dde55_seed_0
ta_gap_greedy 17.13
ta_gap_samp 12.72
la_gap_greedy 10.69
la_gap_samp 5.2
dmu_gap_greedy 28.13
dmu_gap_samp 22.86
folder C:\Users\jesse\PycharmProjects\offline_fjsp\offline-fjsp\collect_results\checkpoints_exp_jsp\jsp_dataset_exp_15_10\CDQAC\sd1\dispatching-bf43fea9_seed_1
ta_gap_greedy 17.31
ta_gap_samp 12.99
la_gap_greedy 11.38
la_gap_samp 5.69
dmu_gap_greedy 28.95
dmu_gap_samp 24.03
folder C:\Users\jesse\PycharmProjects\offline_fjsp\offline-fjsp\collect_results\checkpoints_exp_jsp\jsp_dataset_exp_15_10\CDQAC\sd1\dispatching-c9b2a0be_seed_3
ta_gap_greedy 17.4

In [35]:
import numpy as np

def get_grouped_df(folder_path: str, num_runs="_100"):
    dmu_df = pd.read_csv(os.path.join(folder_path, f"DMU{num_runs}_new.csv"))
    # mask = (df_opt == dmu_df["name"]).all(axis=1)

    # print(dmu_df.groupby(["n_j", "n_m"]).mean(numeric_only=True))
    la_df = pd.read_csv(os.path.join(folder_path, f"LA{num_runs}_new.csv"))
    ta_df = pd.read_csv(os.path.join(folder_path, f"TA{num_runs}_new.csv"))
    dmu_df_grouped = dmu_df.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    la_df_grouped = la_df.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    ta_df_grouped = ta_df.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    return dmu_df_grouped, la_df_grouped, ta_df_grouped

def gather_results_method_grouped(path_folder: str | Path, num_runs="_100") -> dict[str, List]:
    """
    Gather results from all folders in the given path.

    Parameters
    ----------
    path_folder : str | Path
        Path to the directory containing the folders.
    method : str
        Method to gather results for.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary with folder names as keys and DataFrames as values.
    """
    path_folder = os.path.join(path_folder, "sd1")

    path_folder = Path(path_folder).expanduser().resolve()
    if not path_folder.is_dir():
        raise NotADirectoryError(f"{path_folder} is not a directory.")

    folder_dict = group_folders_by_prefix(path_folder)
    results = {}


    for exp_name, folders in folder_dict.items():
        results[exp_name] = {}
        dmu_df_list = []
        la_df_list = []
        ta_df_list = []
        assert len(folders) == 4, f"Expected 4 folders for {exp_name}, but found {len(folders)}"
        for folder in folders:
            n_better = 0
            dmu_grouped_df, la_grouped_df, ta_grouped_df = get_grouped_df(folder, num_runs)
            dmu_df_list.append(dmu_grouped_df)
            la_df_list.append(la_grouped_df)
            ta_df_list.append(ta_grouped_df)


            # if mk_gap_greedy < 12.97:
            #     n_better += 1
            # if mk_gap_samp < 8.95:
            #     n_better += 1
            # if rdata_gap_greedy < 11.15:
            #     n_better += 1
            # if rdata_gap_samp < 4.95:
            #     n_better += 1
            # if edata_gap_greedy < 14.41:
            #     n_better += 1
            # if edata_gap_samp < 8.17:
            #     n_better += 1
            # if vdata_gap_greedy < 3.28:
            #     n_better += 1
            # if vdata_gap_samp < 0.69:
            #     n_better += 1
            # if large:
            #     if gen_gap_greedy < 12.42:
            #         n_better += 1
            #     if gen_gap_samp < 6.79:
            #         n_better += 1
            # else:
            #
            #     if gen_gap_greedy < 10.87:
            #         n_better += 1
            #     if gen_gap_samp < 5.57:
            #         n_better += 1
            # print(f"Folder: {folder}, n_better: {n_better}")
        data_dmu = np.stack([dmu_df.values for dmu_df in dmu_df_list])
        data_la = np.stack([la_df.values for la_df in la_df_list])
        data_ta = np.stack([ta_df.values for ta_df in ta_df_list])

        mean_df_dmu = pd.DataFrame(data_dmu.mean(axis=0), columns=dmu_grouped_df.columns, index=dmu_grouped_df.index)
        std_df_dmu = pd.DataFrame(data_dmu.std(axis=0, ddof=1), columns=dmu_grouped_df.columns, index=dmu_grouped_df.index)
        comb_df_dmu = mean_df_dmu.join(std_df_dmu, lsuffix="_mean", rsuffix="_std")
        comb_df_dmu = comb_df_dmu[["gap_det_offline_mean", "gap_det_offline_std", "runtime_det_offline_mean", "runtime_det_offline_std", "gap_stoch_offline_mean", "gap_stoch_offline_std", "runtime_stoch_offline_mean", "runtime_stoch_offline_std"]]

        mean_df_la = pd.DataFrame(data_la.mean(axis=0), columns=la_grouped_df.columns, index=la_grouped_df.index)
        std_df_la = pd.DataFrame(data_la.std(axis=0, ddof=1), columns=la_grouped_df.columns, index=la_grouped_df.index)
        comb_df_la = mean_df_la.join(std_df_la, lsuffix="_mean", rsuffix="_std")
        comb_df_la = comb_df_la[["gap_det_offline_mean", "gap_det_offline_std", "runtime_det_offline_mean", "runtime_det_offline_std", "gap_stoch_offline_mean", "gap_stoch_offline_std", "runtime_stoch_offline_mean", "runtime_stoch_offline_std"]]
        mean_df_ta = pd.DataFrame(data_ta.mean(axis=0), columns=ta_grouped_df.columns, index=ta_grouped_df.index)
        std_df_ta = pd.DataFrame(data_ta.std(axis=0, ddof=1), columns=ta_grouped_df.columns, index=ta_grouped_df.index)
        comb_df_ta = mean_df_ta.join(std_df_ta, lsuffix="_mean", rsuffix="_std")
        comb_df_ta = comb_df_ta[["gap_det_offline_mean", "gap_det_offline_std", "runtime_det_offline_mean", "runtime_det_offline_std", "gap_stoch_offline_mean", "gap_stoch_offline_std", "runtime_stoch_offline_mean", "runtime_stoch_offline_std"]]
        # comb_df_ta['group'] = ta_df_list

        results[exp_name]["dmu"] = comb_df_dmu
        results[exp_name]["la"] = comb_df_la
        results[exp_name]["ta"] = comb_df_ta

    return results

def get_val_greedy(row_df, round_val=2):
    row_df = row_df.round(round_val)
    row_df_mean = row_df["gap_det_offline_mean"]
    row_df_std = row_df["gap_det_offline_std"]
    return f"{row_df_mean}$\pm${row_df_std}"

def get_val_stoch(row_df, round_val=2):
    row_df = row_df.round(round_val)
    row_df_mean = row_df["gap_stoch_offline_mean"]
    row_df_std = row_df["gap_stoch_offline_std"]
    return f"{row_df_mean}$\pm${row_df_std}"

def get_mean_total(res_10_5, res_15_10, dataset):


    disp_res_10_5 = res_10_5["dispatching"][dataset].mean()
    # print(disp_res_10_5)
    disp_res_15_10 = res_15_10["dispatching"][dataset].mean()
    pop_res_10_5 = res_10_5["population"][dataset].mean()
    pop_res_15_10 = res_15_10["population"][dataset].mean()
    dis_pop_res_10_5 = res_10_5["dis_pop"][dataset].mean()
    dis_pop_res_15_10 = res_15_10["dis_pop"][dataset].mean()
    random_res_10_5 = res_10_5["random"][dataset].mean()
    random_res_15_10 = res_15_10["random"][dataset].mean()
    string_val = ""
    string_val += get_val_greedy(disp_res_10_5) + " & "
    string_val += get_val_greedy(pop_res_10_5) + " & "
    string_val += get_val_greedy(dis_pop_res_10_5) + " & "
    string_val += get_val_greedy(random_res_10_5) + " & "
    string_val += get_val_greedy(disp_res_15_10) + " & "
    string_val += get_val_greedy(pop_res_15_10) + " & "
    string_val += get_val_greedy(dis_pop_res_15_10) + " & "
    string_val += get_val_greedy(random_res_15_10) + " & "
    string_val += get_val_stoch(disp_res_10_5) + " & "
    string_val += get_val_stoch(pop_res_10_5) + " & "
    string_val += get_val_stoch(dis_pop_res_10_5) + " & "
    string_val += get_val_stoch(random_res_10_5) + " & "
    string_val += get_val_stoch(disp_res_15_10) + " & "
    string_val += get_val_stoch(pop_res_15_10) + " & "
    string_val += get_val_stoch(dis_pop_res_15_10) + " & "
    string_val += get_val_stoch(random_res_15_10) + " \\\\"
    print(string_val)

def get_res_df(res_10_5, res_15_10, n_j, n_m, dataset):
    string_val = ""
    disp_res_10_5 = res_10_5["dispatching"][dataset].loc[(n_j, n_m)]
    disp_res_15_10 = res_15_10["dispatching"][dataset].loc[(n_j, n_m)]
    pop_res_10_5 = res_10_5["population"][dataset].loc[(n_j, n_m)]
    pop_res_15_10 = res_15_10["population"][dataset].loc[(n_j, n_m)]
    dis_pop_res_10_5 = res_10_5["dis_pop"][dataset].loc[(n_j, n_m)]
    dis_pop_res_15_10 = res_15_10["dis_pop"][dataset].loc[(n_j, n_m)]
    random_res_10_5 = res_10_5["random"][dataset].loc[(n_j, n_m)]
    random_res_15_10 = res_15_10["random"][dataset].loc[(n_j, n_m)]

    string_val += get_val_greedy(disp_res_10_5) + " & "
    string_val += get_val_greedy(pop_res_10_5) + " & "
    string_val += get_val_greedy(dis_pop_res_10_5) + " & "
    string_val += get_val_greedy(random_res_10_5) + " & "
    string_val += get_val_greedy(disp_res_15_10) + " & "
    string_val += get_val_greedy(pop_res_15_10) + " & "
    string_val += get_val_greedy(dis_pop_res_15_10) + " & "
    string_val += get_val_greedy(random_res_15_10) + " & "
    string_val += get_val_stoch(disp_res_10_5) + " & "
    string_val += get_val_stoch(pop_res_10_5) + " & "
    string_val += get_val_stoch(dis_pop_res_10_5) + " & "
    string_val += get_val_stoch(random_res_10_5) + " & "
    string_val += get_val_stoch(disp_res_15_10) + " & "
    string_val += get_val_stoch(pop_res_15_10) + " & "
    string_val += get_val_stoch(dis_pop_res_15_10) + " & "
    string_val += get_val_stoch(random_res_15_10) + " \\\\"
    print(string_val)





    # pass

def get_results_row(dataset="ta"):
    results_15_10 = gather_results_method_grouped("./checkpoints_exp_jsp/jsp_dataset_exp_15_10/CDQAC")
    results_10_5 = gather_results_method_grouped("./checkpoints_exp_jsp/jsp_dataset_exp_10_5/CDQAC", num_runs="")
    index_list = results_10_5["random"][dataset].index.values
    for n_j, n_m in index_list:
        print(f"\nn_j: {n_j}, n_m: {n_m}")
        get_res_df(results_10_5, results_15_10, n_j, n_m, dataset)

    print("\nMean results")
    get_mean_total(results_10_5, results_15_10, dataset)
        # return




get_results_row(dataset="dmu")




n_j: 20, n_m: 15
24.87$\pm$1.51 & 24.03$\pm$0.94 & 24.47$\pm$2.11 & 24.49$\pm$1.83 & 27.13$\pm$0.74 & 26.03$\pm$1.0 & 24.94$\pm$1.91 & 26.05$\pm$1.37 & 19.4$\pm$0.63 & 19.29$\pm$0.94 & 19.63$\pm$0.81 & 18.82$\pm$0.86 & 20.23$\pm$0.8 & 19.4$\pm$1.05 & 19.5$\pm$1.3 & 19.59$\pm$0.72 \\

n_j: 20, n_m: 20
23.3$\pm$0.36 & 21.29$\pm$1.19 & 22.01$\pm$1.12 & 21.71$\pm$1.47 & 24.01$\pm$0.5 & 22.86$\pm$1.19 & 22.73$\pm$2.13 & 22.67$\pm$1.4 & 17.66$\pm$0.45 & 17.62$\pm$1.15 & 18.03$\pm$0.54 & 17.13$\pm$0.71 & 17.59$\pm$0.62 & 17.4$\pm$0.62 & 17.6$\pm$0.66 & 17.23$\pm$0.68 \\

n_j: 30, n_m: 15
29.63$\pm$0.69 & 28.22$\pm$1.8 & 28.71$\pm$2.63 & 28.76$\pm$1.72 & 30.3$\pm$1.13 & 29.66$\pm$1.54 & 29.19$\pm$1.49 & 29.15$\pm$1.19 & 24.21$\pm$0.61 & 23.22$\pm$1.1 & 24.2$\pm$1.21 & 23.67$\pm$1.7 & 25.93$\pm$1.37 & 24.04$\pm$1.21 & 24.05$\pm$1.9 & 24.25$\pm$0.87 \\

n_j: 30, n_m: 20
28.72$\pm$1.13 & 28.33$\pm$1.0 & 28.53$\pm$2.57 & 28.6$\pm$2.39 & 30.43$\pm$1.04 & 28.65$\pm$1.32 & 28.5$\pm$2.35 & 28.24$\pm$

In [16]:
df_res_disp_dmu = pd.read_csv("./jsp_solutions/dmu_disp_results.csv")
df_res_disp_tai = pd.read_csv("./jsp_solutions/ta_disp_results.csv")
df_res_mwr = df_res_disp_dmu[df_res_disp_tai["methods"] == "MOPNR"]
mean_gap_mwr = df_res_mwr["gap"].mean()
df_res_mwr_group = df_res_mwr.groupby(["n_j", "n_m"]).mean(numeric_only=True)
print(mean_gap_mwr)
df_res_mwr_group.round(2)

33.72611931580539


makespan    gap  time
n_j n_m                       
20  15     3960.8  30.26  0.24
    20     4423.6  26.86  0.38
30  15     5337.8  36.36  0.42
    20     5735.7  33.66  0.67
40  15     6668.1  35.47  0.63
    20     7189.8  35.87  1.18
50  15     8088.3  34.81  0.84
    20     8594.6  36.51  1.61

In [27]:
df_results_prev_paper_ta = pd.read_csv("./offline_ld_res/tai_all.csv")
df_results_prev_paper_dmu = pd.read_csv("./offline_ld_res/dmu_all.csv")

# df_results_prev_paper_ta
mean_tai_l2d = df_results_prev_paper_ta["dqn_noisy"].mean().round(1)
df_results_prev_paper_ta_group = df_results_prev_paper_ta.groupby(["n_j", "n_m"]).mean(numeric_only=True)
res_l2d = df_results_prev_paper_ta_group["dqn_noisy"].values.round(1).tolist()

res_l2d += [float(mean_tai_l2d)]
print(res_l2d)

mean_dmu_l2d = df_results_prev_paper_dmu["dqn_noisy"].mean().round(1)
df_results_prev_paper_dmu_group = df_results_prev_paper_dmu.groupby(["n_j", "n_m"]).mean(numeric_only=True)
res_l2d += df_results_prev_paper_dmu_group["dqn_noisy"].values.round(1).tolist()
res_l2d += [float(mean_dmu_l2d)]
print(res_l2d)

[25.8, 30.2, 28.9, 29.2, 33.1, 20.6, 24.3, 12.7, 25.6]
[25.8, 30.2, 28.9, 29.2, 33.1, 20.6, 24.3, 12.7, 25.6, 35.8, 32.8, 38.8, 36.0, 35.5, 38.5, 34.1, 38.9, 36.3]


In [17]:
def get_results_cdqac_det(path):
    df_ta = pd.read_csv(os.path.join(path, "TA_128_new.csv"))
    df_dmu = pd.read_csv(os.path.join(path, "DMU_128_new.csv"))

    mean_gap_det_ta = df_ta["gap_det_offline"].mean().round(1)
    df_ta_group = df_ta.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    res_ta = df_ta_group["gap_det_offline"].values.round(1).tolist()
    res_ta += [float(mean_gap_det_ta)]
    mean_gap_det_dmu = df_dmu["gap_det_offline"].mean().round(1)
    df_dmu_group = df_dmu.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    res_dmu = df_dmu_group["gap_det_offline"].values.round(1).tolist()
    res_dmu += [float(mean_gap_det_dmu)]
    res_ta += df_dmu_group["gap_det_offline"].values.round(1).tolist()
    res_ta += [float(mean_gap_det_dmu)]
    # print(mean_gap_det_ta, mean_gap_det_dmu)
    print(res_ta)

get_results_cdqac_det("./checkpoints_test_fjsp_jsp/10_5/CDQAC/sd1/random-701eafec_seed_1")


[16.1, 18.6, 18.5, 19.7, 22.3, 13.9, 13.5, 10.1, 16.6, 23.5, 22.3, 27.5, 27.0, 23.4, 25.8, 21.4, 25.0, 24.5]


In [18]:
def get_results_cdqac_stoch(path):
    df_ta = pd.read_csv(os.path.join(path, "TA_128_new.csv"))
    df_dmu = pd.read_csv(os.path.join(path, "DMU_128_new.csv"))

    mean_gap_det_ta = df_ta["gap_stoch_offline"].mean().round(1)
    df_ta_group = df_ta.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    res_ta = df_ta_group["gap_stoch_offline"].values.round(1).tolist()
    res_ta += [float(mean_gap_det_ta)]
    mean_gap_det_dmu = df_dmu["gap_stoch_offline"].mean().round(1)
    df_dmu_group = df_dmu.groupby(["n_j", "n_m"]).mean(numeric_only=True)
    res_dmu = df_dmu_group["gap_stoch_offline"].values.round(1).tolist()
    res_dmu += [float(mean_gap_det_dmu)]
    res_ta += df_dmu_group["gap_stoch_offline"].values.round(1).tolist()
    res_ta += [float(mean_gap_det_dmu)]
    # print(df_ta)
    print(res_ta)

get_results_cdqac_stoch("./checkpoints_exp_jsp/jsp_dataset_exp_10_5/CDQAC/sd1/random-f1b899ce_seed_1")

[10.1, 13.4, 12.8, 14.9, 16.8, 9.7, 10.6, 3.6, 11.5, 18.2, 16.1, 22.1, 22.9, 20.3, 23.7, 21.4, 25.2, 21.2]
